In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from xgboost import XGBClassifier

data = pd.read_csv("vehicle_damage_dataset.csv")

data


,view_type,angle,part,damage,airbag,severity
0,external,left,Battery Tester,lamp broken,not_applicable,moderate
1,internal,rear_right,Tire,none,not_applicable,undamaged
2,internal,front_right,Screen,dent,not_applicable,moderate
3,internal,front_right,Wiper,tire flat,not_applicable,moderate
4,internal,rear_right,Battery Tester,none,not_applicable,undamaged
...,...,...,...,...,...,...
995,external,front,Gauge Manual,dent,not_applicable,moderate
996,internal,front_right,Screen,tire flat,not_applicable,moderate
997,internal,rear,Battery Tester,dent,not_applicable,moderate
998,external,rear_left,Screen,none,not_applicable,undamaged


In [2]:
print("Dataset shape:", data.shape)
print("Columns:", data.columns)

X = data.drop(columns=['severity'])  
y = data['severity'] 

label_encoders = {}
for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        label_encoders[col] = le


target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


Dataset shape: (1000, 6)
Columns: Index(['view_type', 'angle', 'part', 'damage', 'airbag', 'severity'], dtype='object')


In [3]:
for col, le in label_encoders.items():
    print(f"Column: {col}")
    print("Classes:", le.classes_)
    mapping = {index: label for index, label in enumerate(le.classes_)}
    print("Mapping (encoded -> original):", mapping)
    print("-" * 40)

Column: view_type
Classes: ['external' 'internal']
Mapping (encoded -> original): {0: 'external', 1: 'internal'}
----------------------------------------
Column: angle
Classes: ['front' 'front_left' 'front_right' 'left' 'rear' 'rear_left' 'rear_right'
 'right']
Mapping (encoded -> original): {0: 'front', 1: 'front_left', 2: 'front_right', 3: 'left', 4: 'rear', 5: 'rear_left', 6: 'rear_right', 7: 'right'}
----------------------------------------
Column: part
Classes: ['Battery Tester' 'Cabin Air Filter' 'Gauge Brakes' 'Gauge Dial'
 'Gauge Digital' 'Gauge Manual' 'Headlight' 'License Plate' 'Screen'
 'Tire' 'Wiper']
Mapping (encoded -> original): {0: 'Battery Tester', 1: 'Cabin Air Filter', 2: 'Gauge Brakes', 3: 'Gauge Dial', 4: 'Gauge Digital', 5: 'Gauge Manual', 6: 'Headlight', 7: 'License Plate', 8: 'Screen', 9: 'Tire', 10: 'Wiper'}
----------------------------------------
Column: damage
Classes: ['crack' 'dent' 'glass shatter' 'lamp broken' 'none' 'scratch' 'tire flat']
Mapping (enco

In [4]:
import pickle
with open('feature_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)
with open('severity_encoder.pkl', 'wb') as f:
    pickle.dump(target_encoder, f)

In [5]:
model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [6]:
y_pred = model.predict(X_test)

In [7]:
model.save_model("xgb_model.json")

In [8]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted') 
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)

Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1-score : 1.0
